In [2]:
from datetime import timedelta
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import auth
from google.cloud import bigquery, storage

# Authenticate and set up environment
auth.authenticate_user()
project_id = 'mcxrp-429811'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
!gcloud config set project {project_id}

# Initialize BigQuery client
client = bigquery.Client(project=project_id)

# Initialize GCP Storage client with user_project parameter
storage_client = storage.Client(project=project_id)
bucket_name = 'cxr_embedding'
bucket = storage_client.bucket(bucket_name, user_project=project_id)

# Path to the data in your GCP bucket
data_path = 'image-embeddings-mimic-cxr-1.0.physionet.org/files'

# Query BigQuery to get required columns
query = """
SELECT study.study_id, study.subject_id, study.study_datetime, study.path, record_list.dicom_id
FROM `physionet-data.mimic_cxr.study` AS study
JOIN `physionet-data.mimic_cxr.record_list` AS record_list
ON study.study_id = record_list.study_id
"""
query_job = client.query(query)
bigquery_df = query_job.to_dataframe()

# Load the ARDS cohort data table from GCP bucket
ards_data_path = 'ARDS cohort ~1000 patients 28 days July.csv'
ards_blob = bucket.blob(ards_data_path)
ards_blob.download_to_filename('/tmp/ards_data.csv')
ards_df = pd.read_csv('/tmp/ards_data.csv')

# Filter ARDS cohort data to ensure one row per patient at day zero
ards_filtered_df = ards_df[ards_df['days_from_start'] == 0]

# Select required columns from ARDS cohort data
ards_filtered_df = ards_filtered_df[['subject_id', 'timezero', 'icu_mort', 'avg_peep']]

# Convert time columns to datetime format
bigquery_df['study_datetime'] = pd.to_datetime(bigquery_df['study_datetime'])
ards_filtered_df['timezero'] = pd.to_datetime(ards_filtered_df['timezero'])

# Keep only the rows where subject_id is in both dataframes
merged_subject_ids = set(bigquery_df['subject_id']).intersection(set(ards_filtered_df['subject_id']))
bigquery_df = bigquery_df[bigquery_df['subject_id'].isin(merged_subject_ids)]
ards_filtered_df = ards_filtered_df[ards_filtered_df['subject_id'].isin(merged_subject_ids)]


Updated property [core/project].


<ipython-input-2-4693be1e8ac2>:40: DtypeWarning: Columns (41) have mixed types. Specify dtype option on import or set low_memory=False.
  ards_df = pd.read_csv('/tmp/ards_data.csv')


In [ ]:
# Function to load embedding vectors from GCP bucket
def load_embedding_vector(file_path, dicom_id):
    clean_path = file_path.replace('.txt', '')
    if clean_path.startswith('files/'):
        clean_path = clean_path[len('files/'):]
    file_name = dicom_id + '.tfrecord'
    full_path = os.path.join(data_path, clean_path, file_name)
    blob = bucket.blob(full_path)

    #print(f"Checking if blob exists at path: {full_path}")  # Debugging log

    if blob.exists():
        #print(f"Blob found. Downloading data from: {full_path}")  # Debugging log
        try:
            raw_data = blob.download_as_bytes()
            print(f"Raw data size: {len(raw_data)} bytes")  # Debugging log

            # Save the downloaded data to a temporary file for TensorFlow to read
            temp_file_path = '/tmp/temp.tfrecord'
            with open(temp_file_path, 'wb') as f:
                f.write(raw_data)

            raw_dataset = tf.data.TFRecordDataset(temp_file_path)
            for raw_record in raw_dataset.take(1):
                example = tf.train.Example()
                example.ParseFromString(raw_record.numpy())
                embedding = example.features.feature['embedding'].float_list.value
                return embedding
        except tf.errors.DataLossError as e:
            print(f"Data loss error reading TFRecord file {full_path}: {e}")
        except tf.errors.InvalidArgumentError as e:
            print(f"Invalid argument error reading TFRecord file {full_path}: {e}")
        except Exception as e:
            print(f"General error reading TFRecord file {full_path}: {e}")
    else:
        print(f"File {full_path} does not exist")
    return None

# Create a new DataFrame to hold valid records
valid_rows = []

# Load embeddings into the DataFrame
for index, row in bigquery_df.iterrows():
    path = row['path']
    dicom_id = row['dicom_id']
    embedding = load_embedding_vector(path, dicom_id)
    if embedding:
        for i, val in enumerate(embedding):
            row[f'embedding_{i}'] = val
        valid_rows.append(row)

# Create a new DataFrame from valid rows
valid_df = pd.DataFrame(valid_rows)

# Merge the two dataframes on subject_id
merged_df = pd.merge(valid_df, ards_filtered_df, on='subject_id', how='inner')

# Calculate the time difference between study_datetime and timezero
merged_df['time_diff'] = merged_df['study_datetime'] - merged_df['timezero']

# Filter out rows where the study_datetime is not within 1 day before or within 2 days after the timezero
filtered_df = merged_df[(merged_df['time_diff'] >= timedelta(days=-1)) & (merged_df['time_diff'] <= timedelta(days=2))].copy()

# Sort by subject_id and the absolute value of time_diff, then drop duplicates keeping the closest study
filtered_df.loc[:, 'abs_time_diff'] = filtered_df['time_diff'].abs()
filtered_df = filtered_df.sort_values(by=['subject_id', 'abs_time_diff']).drop_duplicates(subset='subject_id', keep='first')

# Drop the auxiliary column
filtered_df = filtered_df.drop(columns=['abs_time_diff'])

# Reorder columns to have the embedding columns at the end
embedding_columns = [col for col in filtered_df.columns if col.startswith('embedding_')]
non_embedding_columns = [col for col in filtered_df.columns if not col.startswith('embedding_')]

final_columns = non_embedding_columns + embedding_columns
final_df = filtered_df[final_columns]

# Save the final DataFrame to a CSV file in the GCP bucket
final_combined_data_path = 'filtered_data1.csv'
final_combined_blob = bucket.blob(final_combined_data_path)
final_combined_blob.upload_from_string(final_df.to_csv(index=False), 'text/csv')
print(f"Final combined data saved to {final_combined_data_path} in GCP bucket")

Streaming output truncated to the last 5000 lines.
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p15/p15084163/s51826003/497ddd1f-2ddafc35-888245e2-932ec087-af8fedca.tfrecord does not exist
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p15/p15084163/s51826003/512250e4-e5b59f41-5d252ba1-0f50952c-683b9dcd.tfrecord does not exist
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p15/p15084163/s52035922/a6792208-52797ab4-59c75f40-56eb0da0-6cbe3a03.tfrecord does not exist
Raw data size: 5698 bytes
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p15/p15084163/s52433897/8c72b72f-cd44f020-c2d149de-49d5d9bb-5150ff28.tfrecord does not exist
Raw data size: 5698 bytes
Raw data size: 5698 bytes
File image-embeddings-mimic-cxr-1.0.physi

In [ ]:
## For filtering rows out where the time difference between X-ray taken and time_zero where the peep level is measured is too large (<3 days)
from datetime import timedelta
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import auth
from google.cloud import bigquery, storage

# Authenticate and set up environment
auth.authenticate_user()
project_id = 'mcxrp-429811'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
!gcloud config set project {project_id}

# Initialize BigQuery client
client = bigquery.Client(project=project_id)

# Initialize GCP Storage client with user_project parameter
storage_client = storage.Client(project=project_id)
bucket_name = 'cxr_embedding'
bucket = storage_client.bucket(bucket_name, user_project=project_id)

# Path to the final combined data CSV in your GCP bucket
final_combined_data_path = 'final_combined_data1.csv'
final_combined_blob = bucket.blob(final_combined_data_path)

# Download the final combined data CSV to a local temporary file
local_combined_data_path = '/tmp/final_combined_data.csv'
final_combined_blob.download_to_filename(local_combined_data_path)

# Load the final combined data CSV into a DataFrame
final_df = pd.read_csv(local_combined_data_path)

# Ensure 'time_diff' is in timedelta format
final_df['time_diff'] = pd.to_timedelta(final_df['time_diff'])

# Filter out rows where the time_diff is not between 24 hours and 48 hours
filtered_df = final_df[(final_df['time_diff'] > timedelta(hours=24)) & (final_df['time_diff'] <= timedelta(hours=48))]


# Save the filtered DataFrame to a CSV file in the GCP bucket
filtered_data_path = 'filtered_combined_data.csv'
filtered_blob = bucket.blob(filtered_data_path)
filtered_blob.upload_from_string(filtered_df.to_csv(index=False), 'text/csv')

print(f"Filtered combined data saved to {filtered_data_path} in GCP bucket")


In [27]:
## Now, let's do dimensionality reduction then logistic regression to find relation ship between x-rays and peep levels + mortality rate
## Go down for just peep and x-rays
# Path to the filtered combined data CSV in your GCP bucket
filtered_data_path = 'filtered_combined_data.csv'
filtered_blob = bucket.blob(filtered_data_path)

# Download the filtered combined data CSV to a local temporary file
local_filtered_data_path = '/tmp/filtered_combined_data.csv'
filtered_blob.download_to_filename(local_filtered_data_path)

# Load the filtered combined data CSV into a DataFrame
df = pd.read_csv(local_filtered_data_path)

# Display the first few rows of the dataframe to verify the data
print(df.head())
# First, let's do PCA
from sklearn.decomposition import PCA

# Extract embedding vectors
embedding_columns = [col for col in df.columns if col.startswith('embedding_')]
embeddings = df[embedding_columns]

# Apply PCA
pca = PCA(n_components=50)  # Reduce to 50 components or adjust based on explained variance
embeddings_pca = pca.fit_transform(embeddings)

# Add PCA components to the dataframe
for i in range(embeddings_pca.shape[1]):
    df[f'pca_{i+1}'] = embeddings_pca[:, i]

# Display the explained variance ratio
print(pca.explained_variance_ratio_)

   study_id  subject_id           study_datetime  \
0  52515667    10021487  2116-12-04 04:57:57.039   
1  56863750    10021927  2180-09-20 15:46:21.234   
2  57716752    10051043  2192-10-09 20:26:04.984   
3  58771416    10110764  2130-09-14 04:24:35.062   
4  50996857    10111112  2150-02-15 21:56:48.796   

                                path  \
0  files/p10/p10021487/s52515667.txt   
1  files/p10/p10021927/s56863750.txt   
2  files/p10/p10051043/s57716752.txt   
3  files/p10/p10110764/s58771416.txt   
4  files/p10/p10111112/s50996857.txt   

                                       dicom_id             timezero  \
0  9f7d78ea-3678f7f5-ad9613bc-7e4a779d-b7384021  2116-12-03 05:10:00   
1  00f6367a-bb53009c-e373833e-14201ddd-5edb8f5f  2180-09-20 16:33:00   
2  3abd6e9e-609cfd2a-2d3bda3a-09b013a0-80520892  2192-10-09 15:48:00   
3  dad5aca3-cd745760-e9ab74fa-5c07a3d1-75b4def3  2130-09-14 01:28:00   
4  9055c8de-1400b556-7de439ff-8d668884-122246cd  2150-02-15 21:48:00   

   icu_mort  

In [29]:
# Second, let's do t-sne reduction
from sklearn.manifold import TSNE

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42)  # Reduce to 2 dimensions for visualization
embeddings_tsne = tsne.fit_transform(embeddings_pca)

# Add t-SNE components to the dataframe
df['tsne_1'] = embeddings_tsne[:, 0]
df['tsne_2'] = embeddings_tsne[:, 1]

# Display the first few rows of the dataframe with the new columns
print(df[['tsne_1', 'tsne_2']].head())


      tsne_1     tsne_2
0 -15.995447  15.247846
1   7.715063  -2.015188
2   7.368937   1.864384
3  -8.776114  -6.767620
4   4.968107  -0.755686


In [ ]:
# Now, regression model!

import statsmodels.api as sm
import numpy as np

# Prepare the data for regression analysis
X = df[['avg_peep'] + [f'pca_{i+1}' for i in range(50)]]
y = df['icu_mort']

# Add a constant to the independent variables
X = sm.add_constant(X)

# Fit the logistic regression model
model = sm.Logit(y, X)
result = model.fit()

# Display the summary of the regression analysis
print(result.summary())

# Exponentiate the coefficients to interpret them as odds ratios
odds_ratios = np.exp(result.params)
print(odds_ratios)


In [16]:
from datetime import timedelta
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import auth
from google.cloud import bigquery, storage

# Authenticate and set up environment
auth.authenticate_user()
project_id = 'mcxrp-429811'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
!gcloud config set project {project_id}

# Initialize BigQuery client
client = bigquery.Client(project=project_id)

# Initialize GCP Storage client with user_project parameter
storage_client = storage.Client(project=project_id)
bucket_name = 'cxr_embedding'
bucket = storage_client.bucket(bucket_name, user_project=project_id)

# Path to the data in your GCP bucket
data_path = 'image-embeddings-mimic-cxr-1.0.physionet.org/files'

# Query BigQuery to get required columns
query = """
SELECT study.study_id, study.subject_id, study.study_datetime, study.path, record_list.dicom_id
FROM `physionet-data.mimic_cxr.study` AS study
JOIN `physionet-data.mimic_cxr.record_list` AS record_list
ON study.study_id = record_list.study_id
"""
query_job = client.query(query)
bigquery_df = query_job.to_dataframe()

# Load the ARDS cohort data table from GCP bucket
ards_data_path = 'ARDS cohort ~1000 patients 28 days July.csv'
ards_blob = bucket.blob(ards_data_path)
ards_blob.download_to_filename('/tmp/ards_data.csv')
ards_df = pd.read_csv('/tmp/ards_data.csv', low_memory=False)

# Select required columns from ARDS cohort data
ards_filtered_df = ards_df[['subject_id', 'days_from_start', 'timezero', 'icu_mort', 'avg_peep']]

# Convert time columns to datetime format
bigquery_df['study_datetime'] = pd.to_datetime(bigquery_df['study_datetime'])
ards_filtered_df.loc[:, 'timezero'] = pd.to_datetime(ards_filtered_df['timezero'])

# Keep only the rows where subject_id is in both dataframes
merged_subject_ids = set(bigquery_df['subject_id']).intersection(set(ards_filtered_df['subject_id']))
bigquery_df = bigquery_df[bigquery_df['subject_id'].isin(merged_subject_ids)]
ards_filtered_df = ards_filtered_df[ards_filtered_df['subject_id'].isin(merged_subject_ids)]

# Merge the two dataframes on subject_id
merged_df = pd.merge(bigquery_df, ards_filtered_df, on='subject_id', how='inner')

# Calculate the time difference between study_datetime and timezero in hours
merged_df['time_diff_hours'] = (merged_df['study_datetime'] - merged_df['timezero']) / np.timedelta64(1, 'h')

# Filter out rows where the time_diff_hours is not within -48 to 24 hours
filtered_df = merged_df[(merged_df['time_diff_hours'] > -48) & (merged_df['time_diff_hours'] <= 24)].copy()

# Sort by subject_id, avg_peep, and the absolute value of time_diff_hours
filtered_df['abs_time_diff_hours'] = filtered_df['time_diff_hours'].abs()
filtered_df = filtered_df.sort_values(by=['subject_id', 'avg_peep', 'abs_time_diff_hours'])

# Drop duplicates based on dicom_id, keeping the row with the closest study_datetime
filtered_df = filtered_df.sort_values(by='abs_time_diff_hours').drop_duplicates(subset='dicom_id', keep='first')

# Drop the auxiliary column
filtered_df = filtered_df.drop(columns=['abs_time_diff_hours'])

# Save the intermediate DataFrame to a local CSV file
local_output_path = '/tmp/test_combined_data.csv'
filtered_df.to_csv(local_output_path, index=False)

# Upload the CSV file to the GCP bucket
blob = bucket.blob('test_combined_data.csv')
blob.upload_from_filename(local_output_path)

print(f"Intermediate combined data saved to GCP bucket at: gs://{bucket_name}/test_combined_data.csv")

# Display the final DataFrame
filtered_df.head()


Updated property [core/project].


<ipython-input-16-a273686d31a1>:58: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  merged_df['time_diff_hours'] = (merged_df['study_datetime'] - merged_df['timezero']) / np.timedelta64(1, 'h')


Intermediate combined data saved to GCP bucket at: gs://cxr_embedding/test_combined_data.csv


,study_id,subject_id,study_datetime,path,dicom_id,days_from_start,timezero,icu_mort,avg_peep,time_diff_hours
67186,52057611,17636206,2111-02-04 06:05:17.859,files/p17/p17636206/s52057611.txt,e720d4df-99ac6ec3-d8c6cb5f-826c8c94-f8e00e90,0,2111-02-04 06:05:00,0,16.400000,0.004961
157277,55050076,15326204,2177-10-29 21:14:08.671,files/p15/p15326204/s55050076.txt,b1473847-85f72ad9-1db89485-b954e0d4-4b0f2037,1,2177-10-29 21:13:00,0,11.000000,0.019075
138648,59278177,18249179,2188-01-26 05:08:28.045,files/p18/p18249179/s59278177.txt,de9342fb-12d69e18-332aeb1f-6393065e-de4aede6,9,2188-01-26 05:07:00,0,3.888889,0.024457
131214,58140351,13826513,2111-10-10 19:00:05.234,files/p13/p13826513/s58140351.txt,564bc007-1233349d-3e648755-9956b293-b303c161,4,2111-10-10 19:02:00,0,5.000000,-0.031879
43749,54368202,12770117,2153-11-02 18:58:30.453,files/p12/p12770117/s54368202.txt,1912e8d7-f8412b97-a02bfe0a-c2d9f680-6222395f,11,2153-11-02 18:56:00,0,7.250000,0.041793


In [ ]:
from datetime import timedelta
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import auth
from google.cloud import bigquery, storage

# Authenticate and set up environment
auth.authenticate_user()
project_id = 'mcxrp-429811'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
!gcloud config set project {project_id}

# Initialize BigQuery client
client = bigquery.Client(project=project_id)

# Initialize GCP Storage client with user_project parameter
storage_client = storage.Client(project=project_id)
bucket_name = 'cxr_embedding'
bucket = storage_client.bucket(bucket_name, user_project=project_id)

# Path to the data in your GCP bucket
data_path = 'image-embeddings-mimic-cxr-1.0.physionet.org/files'

# Query BigQuery to get required columns
query = """
SELECT study.study_id, study.subject_id, study.study_datetime, study.path, record_list.dicom_id
FROM `physionet-data.mimic_cxr.study` AS study
JOIN `physionet-data.mimic_cxr.record_list` AS record_list
ON study.study_id = record_list.study_id
"""
query_job = client.query(query)
bigquery_df = query_job.to_dataframe()

# Load the ARDS cohort data table from GCP bucket
ards_data_path = 'ARDS cohort ~1000 patients 28 days July.csv'
ards_blob = bucket.blob(ards_data_path)
ards_blob.download_to_filename('/tmp/ards_data.csv')
ards_df = pd.read_csv('/tmp/ards_data.csv', low_memory=False)

# Select required columns from ARDS cohort data
ards_filtered_df = ards_df[['subject_id', 'days_from_start', 'timezero', 'icu_mort', 'avg_peep']]

# Convert time columns to datetime format
bigquery_df['study_datetime'] = pd.to_datetime(bigquery_df['study_datetime'])
ards_filtered_df['timezero'] = pd.to_datetime(ards_filtered_df['timezero'])

# Keep only the rows where subject_id is in both dataframes
merged_subject_ids = set(bigquery_df['subject_id']).intersection(set(ards_filtered_df['subject_id']))
bigquery_df = bigquery_df[bigquery_df['subject_id'].isin(merged_subject_ids)]
ards_filtered_df = ards_filtered_df[ards_filtered_df['subject_id'].isin(merged_subject_ids)]

# Merge the two dataframes on subject_id
merged_df = pd.merge(bigquery_df, ards_filtered_df, on='subject_id', how='inner')

# Ensure the time columns are properly aligned and handle NaT values
merged_df['study_datetime'] = pd.to_datetime(merged_df['study_datetime'])
merged_df['timezero'] = pd.to_datetime(merged_df['timezero'])

# Calculate the time difference between study_datetime and timezero in hours
merged_df['time_diff_hours'] = (merged_df['study_datetime'] - merged_df['timezero']).dt.total_seconds() / 3600

# Filter out rows where the time_diff_hours is not within -48 to 24 hours
filtered_df = merged_df[(merged_df['time_diff_hours'] > -48) & (merged_df['time_diff_hours'] <= 24)].copy()

# Sort by subject_id, avg_peep, and the absolute value of time_diff_hours
filtered_df['abs_time_diff_hours'] = filtered_df['time_diff_hours'].abs()
filtered_df = filtered_df.sort_values(by=['subject_id', 'avg_peep', 'abs_time_diff_hours'])

# Drop duplicates based on dicom_id, keeping the row with the closest study_datetime
filtered_df = filtered_df.sort_values(by='abs_time_diff_hours').drop_duplicates(subset='dicom_id', keep='first')

# Function to load embedding vectors from GCP bucket
def load_embedding_vector(file_path, dicom_id):
    clean_path = file_path.replace('.txt', '')
    if clean_path.startswith('files/'):
        clean_path = clean_path[len('files/'):]
    file_name = dicom_id + '.tfrecord'
    full_path = os.path.join(data_path, clean_path, file_name)
    blob = bucket.blob(full_path)

    if blob.exists():
        try:
            raw_data = blob.download_as_bytes()
            # Save the downloaded data to a temporary file for TensorFlow to read
            temp_file_path = '/tmp/temp.tfrecord'
            with open(temp_file_path, 'wb') as f:
                f.write(raw_data)

            raw_dataset = tf.data.TFRecordDataset(temp_file_path)
            for raw_record in raw_dataset.take(1):
                example = tf.train.Example()
                example.ParseFromString(raw_record.numpy())
                embedding = example.features.feature['embedding'].float_list.value
                return embedding
        except Exception as e:
            print(f"Error reading TFRecord file {full_path}: {e}")
    else:
        print(f"File {full_path} does not exist")
    return None

# Create a new DataFrame to hold valid records with embeddings
valid_rows = []

# Load embeddings into the DataFrame
for index, row in filtered_df.iterrows():
    path = row['path']
    dicom_id = row['dicom_id']
    embedding = load_embedding_vector(path, dicom_id)
    if embedding:
        for i, val in enumerate(embedding):
            row[f'embedding_{i}'] = val
        valid_rows.append(row)

# Create a new DataFrame from valid rows
final_df = pd.DataFrame(valid_rows)

# Drop the auxiliary column
final_df = final_df.drop(columns=['abs_time_diff_hours'])

# Save the final DataFrame to a local CSV file
local_output_path = '/tmp/final_combined_data.csv'
final_df.to_csv(local_output_path, index=False)

# Upload the CSV file to the GCP bucket
blob = bucket.blob('final_combined_data.csv')
blob.upload_from_filename(local_output_path)

print(f"Final combined data saved to GCP bucket at: gs://{bucket_name}/final_combined_data.csv")

# Display the final DataFrame
final_df.head()


Updated property [core/project].


<ipython-input-18-82485fc26674>:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ards_filtered_df['timezero'] = pd.to_datetime(ards_filtered_df['timezero'])


File image-embeddings-mimic-cxr-1.0.physionet.org/files/p13/p13299143/s58816141/3203035c-4e0b4005-2ce06b4e-eb92c258-eb485df7.tfrecord does not exist
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p17/p17125981/s50945509/af671a77-9d26b2ee-3bbbb893-84ea8ef4-ea2b6d97.tfrecord does not exist
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p17/p17125981/s53481659/a866bff2-501a4852-e6789b0c-c8ecf737-06ef0889.tfrecord does not exist
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p10/p10165555/s59901062/623937ac-29d5e305-23c46c95-843b275e-2eccbbf0.tfrecord does not exist
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p19/p19654837/s53434468/68406a09-794c2454-4c06f8be-bfc45c5d-8be83afc.tfrecord does not exist
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p10/p10165555/s56658165/033174c1-cd6db741-f1134ede-6a1ba8b5-58b9f5ab.tfrecord does not exist
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p19/p19654837/s57793087/5da94e93-74f5ee0f-acc2f02a

,study_id,subject_id,study_datetime,path,dicom_id,days_from_start,timezero,icu_mort,avg_peep,time_diff_hours,...,embedding_1366,embedding_1367,embedding_1368,embedding_1369,embedding_1370,embedding_1371,embedding_1372,embedding_1373,embedding_1374,embedding_1375
67186,52057611,17636206,2111-02-04 06:05:17.859,files/p17/p17636206/s52057611.txt,e720d4df-99ac6ec3-d8c6cb5f-826c8c94-f8e00e90,0,2111-02-04 06:05:00,0,16.400000,0.004961,...,-0.244843,-0.723355,0.459730,0.684416,1.204422,1.099409,1.627403,-2.043758,1.334290,-0.577311
157277,55050076,15326204,2177-10-29 21:14:08.671,files/p15/p15326204/s55050076.txt,b1473847-85f72ad9-1db89485-b954e0d4-4b0f2037,1,2177-10-29 21:13:00,0,11.000000,0.019075,...,-0.016293,-0.485679,-0.266421,0.954022,1.004177,0.460414,1.543107,-1.965826,0.590041,-1.141346
138648,59278177,18249179,2188-01-26 05:08:28.045,files/p18/p18249179/s59278177.txt,de9342fb-12d69e18-332aeb1f-6393065e-de4aede6,9,2188-01-26 05:07:00,0,3.888889,0.024457,...,-0.164888,-1.578868,0.394493,0.777911,0.924821,1.703792,0.715513,-1.868661,0.472752,-0.443693
131214,58140351,13826513,2111-10-10 19:00:05.234,files/p13/p13826513/s58140351.txt,564bc007-1233349d-3e648755-9956b293-b303c161,4,2111-10-10 19:02:00,0,5.000000,-0.031879,...,-0.361324,-0.771742,-0.131035,0.820588,1.894289,0.918283,0.177461,-1.481192,0.997785,-0.558421
43749,54368202,12770117,2153-11-02 18:58:30.453,files/p12/p12770117/s54368202.txt,1912e8d7-f8412b97-a02bfe0a-c2d9f680-6222395f,11,2153-11-02 18:56:00,0,7.250000,0.041793,...,0.273789,-0.375787,0.408508,0.030080,0.847348,1.879938,0.583067,-1.737626,0.700717,-0.906438
